# Error analysis

This notebook runs the paired backtest and breaks the results down by the four
rubric axes, so you can see not just whether memory helped but *why* a run
succeeded or failed: did it retrieve nothing, retrieve the wrong memory, apply the
right memory badly, or execute invalid actions.

In [ ]:
from datetime import datetime
from pathlib import Path

from ombench.config import Config
from ombench.events.store import EventStore
from ombench.integrations.slack.sync import SlackSync
from ombench.integrations.gcal.sync import GCalSync
from ombench.integrations.gdocs.sync import GDocsSync
from ombench.eval.seed import load_memory_seed
from ombench.eval.tasks import load_tasks
from ombench.eval.runner import BacktestRunner
from ombench.eval.reports import summarize
from ombench.llm.stub import StubLLM
from ombench.storage import open_store
from ombench.timeutil import UTC, FrozenClock

import tempfile
repo = Path('..').resolve()
cfg = Config(home=Path(tempfile.mkdtemp()) / '.ombench', repo_root=repo)
store = open_store(cfg)
es = EventStore(store.backend, store.blobs)
clock = FrozenClock(datetime(2026, 5, 14, 18, 0, 0, tzinfo=UTC))
SlackSync(es, clock=clock, fixtures_path=repo / 'fixtures' / 'slack' / 'workspace.json').run_sync()
GCalSync(es, clock=clock, fixtures_path=repo / 'fixtures' / 'gcal' / 'calendar.json').run_sync()
GDocsSync(es, clock=clock, fixtures_path=repo / 'fixtures' / 'gdocs' / 'docs.json').run_sync()
load_memory_seed(store, repo / 'benchmarks' / 'memory_seed.yaml')
report = BacktestRunner(store, llm=StubLLM()).run(load_tasks(repo / 'benchmarks' / 'tasks'))
summarize(report)


In [ ]:
# Per axis breakdown for the with-memory condition.
for r in report.results:
    wm = r.with_memory.scores
    print(f"{r.task_id:30s} outcome={wm.task_outcome} retrieval={wm.memory_retrieval} "
          f"application={wm.memory_application} validity={wm.action_validity} delta={r.total_delta}")


## Statistics

The supporting statistics are paired: bootstrap confidence intervals on the mean
delta and win rate, and a Wilcoxon signed rank test on the score differences.

In [ ]:
from ombench.eval.stats import bootstrap_mean_ci, wilcoxon_signed_rank, win_rate_ci

deltas = report.deltas()
print('mean delta CI', bootstrap_mean_ci(deltas))
print('win rate CI', win_rate_ci(deltas))
print('wilcoxon', wilcoxon_signed_rank(deltas))
